# 01 — The Agent Loop

> *"Every major AI agent runs the same core loop. A `while` loop that calls an LLM, checks if the response contains tool calls, executes them if it does, and stops if it doesn't."*

**By the end of this notebook you will:**
1. Write the 6-line agent loop from memory
2. Watch it succeed on a normal input
3. Break it on purpose to see real failure modes
4. Add the four production necessities: bounded steps, parallel tool calls, error isolation, observability

Lecture reference: **S8 §3** (*Implementing the Agent Loop*).


## Setup

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
client = AsyncOpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Using model: {MODEL}")

## 1. The 6-line agent loop

This is the entire idea of an LLM agent in 6 lines of meaningful code. Memorise it.

```
while not done:
    msg = llm(messages, tools=tools)        # 1. ask
    messages.append(msg)                    # 2. remember
    if not msg.tool_calls: return msg.text  # 3. natural exit
    for tc in msg.tool_calls:               # 4. act
        result = execute(tc)                # 5. observe
        messages.append(result)             # 6. remember the observation
```

Below is the working version with one knob: `max_steps` to prevent infinite loops.

In [ ]:
async def run_agent_v0(user_msg, tools, executors, max_steps=10):
    """The 6-line agent loop. No safety nets — break it to learn what production needs."""
    messages = [{"role": "user", "content": user_msg}]
    for _ in range(max_steps):
        resp = await client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            result = executors[tc.function.name](**json.loads(tc.function.arguments))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    raise RuntimeError("max_steps exceeded")

## 2. Try it with one tool

We give it a single tool: `add(a, b)`. The model has to *decide* to call this tool to compute `17 + 25` rather than computing it itself in text.

Notice the schema declares `additionalProperties: False` — that's required when you turn on `strict: True`. We'll cover strict mode in detail in notebook 02.


In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "add",
        "description": "Add two integers and return the sum.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}]

EXECUTORS = {"add": lambda a, b: {"sum": a + b}}

answer = await run_agent_v0("What is 17 + 25?", TOOLS, EXECUTORS)
print(answer)

## 3. Break it: hallucinated tool

What happens if the LLM tries to call a tool that doesn't exist?

Ask it to multiply — but we only registered `add`. Watch what happens to your runtime.

In [ ]:
try:
    answer = await run_agent_v0("What is 7 times 8?", TOOLS, EXECUTORS)
    print(answer)
except Exception as e:
    print(f"💥 {type(e).__name__}: {e}")

**The whole agent crashes.** A real production agent must convert tool errors into *observations the LLM can read*, not exceptions that crash the runtime. Then the LLM usually self-corrects on the next turn ("oh, I should use `add` repeatedly").

## 4. Production-hardened loop

We'll add four things:

| Add | Why |
|---|---|
| `parallel_tool_calls=True` | independent tool calls happen in one round-trip |
| `try/except` around tool execution | errors become tool results, not crashes |
| `trace` list | every step is observable — see S8 §7 |
| graceful `tool_choice="none"` final call | when `max_steps` hit, force a text answer |

Other refinements we'll cover later — strict mode (notebook 02), loop fingerprinting and cost caps (notebook 03).

In [ ]:
async def run_agent(user_msg, tools, executors, system=None, max_steps=10, model=None):
    model = model or MODEL
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})
    trace = []

    for step in range(max_steps):
        resp = await client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            parallel_tool_calls=True,
        )
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))
        trace.append({
            "step": step,
            "tokens": resp.usage.total_tokens,
            "tool_calls": [tc.function.name for tc in (msg.tool_calls or [])],
            "content": msg.content,
        })

        # Natural exit — no more tool calls
        if not msg.tool_calls:
            return {"answer": msg.content, "steps": step + 1, "trace": trace, "terminated_by": "natural"}

        # Execute tool calls — convert any error into an observation the LLM can read
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments)
                fn = executors.get(tc.function.name)
                if fn is None:
                    raise KeyError(f"Unknown tool: {tc.function.name}. Available: {list(executors)}")
                result = await fn(**args) if asyncio.iscoroutinefunction(fn) else fn(**args)
                content = json.dumps({"success": True, "data": result})
            except Exception as e:
                content = json.dumps({"success": False, "error": str(e)})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})

    # Max steps hit — force a final text answer instead of raising
    messages.append({
        "role": "user",
        "content": "Max steps reached. Provide your best answer based on what you've learned. Do not call any more tools.",
    })
    final = await client.chat.completions.create(model=model, messages=messages, tool_choice="none")
    return {
        "answer": final.choices[0].message.content,
        "steps": max_steps,
        "trace": trace,
        "terminated_by": "max_steps",
    }

## 5. Re-run the broken case

Same input that crashed `run_agent_v0` — now the agent survives and the LLM gets a chance to recover.

In [ ]:
result = await run_agent(
    "What is 7 times 8? You can only add.",
    TOOLS,
    EXECUTORS,
    system="You are a helpful assistant. Use the available tools.",
)
print("answer:", result["answer"])
print("\ntrace:")
for entry in result["trace"]:
    calls = entry["tool_calls"] or "—"
    print(f"  step {entry['step']}: tools={calls}  tokens={entry['tokens']}")

What you should see: the LLM calls `add` with progressively larger arguments to compute `7×8 = 56`. It treats the missing `multiply` as a constraint and works around it. **That's the LLM self-correcting from a tool-result observation** — exactly what makes ReAct different from plain chain-of-thought.


## 6. Parallel tool calls

When the LLM has *independent* sub-questions it can issue multiple tool calls in a single turn. `parallel_tool_calls=True` lets that happen. Without it you'd get N round-trips for N independent calls.

Watch the `tool_calls` column in the trace below.

In [ ]:
result = await run_agent(
    "Compute 1+1, 2+2, and 3+3 — give me all three answers.",
    TOOLS,
    EXECUTORS,
)
print("answer:", result["answer"])
print("\ntrace:")
for entry in result["trace"]:
    print(f"  step {entry['step']}: {entry['tool_calls']}")

Step 0 should show `['add', 'add', 'add']` — three tool calls in one turn. Step 1 is the final answer.

Without parallel calls you'd see three separate steps each with one call.

## 7. Self-check

Before moving on, can you answer these without scrolling up?

1. Write `run_agent_v0` from memory. (6-ish lines.)
2. Name three things `run_agent_v0` is missing for production.
3. What does `tool_choice="none"` guarantee about the next assistant message?
4. Why do we wrap tool execution in `try/except` instead of letting it raise?
5. What's the difference between `terminated_by="natural"` and `terminated_by="max_steps"`?

## What's next

Notebook **02** turns the loose `TOOLS` and `EXECUTORS` dicts into proper `BaseTool` and `ToolRegistry` classes — and digs into OpenAI's `strict: True` mode (the production must-have for preventing argument-shape hallucinations).
